# Day 6 · Exercise 1: Build a Structured Prompt

**What you'll build:** `build_prompt(instruction: str, context: str, user_input: str, output_format: str) -> str` — a function that assembles the four named parts of a prompt into one string ready to send to a model.

**Why it matters:** Making the four parts explicit in your code means you can diagnose any failing prompt by checking which part is missing or underspecified — no more rewriting from scratch and hoping.

## Your Implementation

In [ ]:
def build_prompt(instruction: str, context: str, user_input: str, output_format: str) -> str:
    """Assemble the four parts of a structured prompt into one string.

    Each part is placed on its own labelled section so the model receives
    clear, unambiguous guidance. The returned string can be passed directly
    as the content of a user or system message.

    Args:
        instruction:   What you want the model to do (the task verb/goal).
        context:       Background the model needs — persona, domain, constraints.
        user_input:    The actual data or text the model should work on.
        output_format: The shape, structure, or style the answer must take.

    Returns:
        A single string with all four parts joined, ready for ollama.chat().

    Example:
        >>> prompt = build_prompt(
        ...     instruction="Classify the sentiment of this review.",
        ...     context="You are a sentiment analyser. Be concise.",
        ...     user_input="The battery life on this phone is absolutely terrible.",
        ...     output_format="Reply with exactly one word: POSITIVE, NEGATIVE, or NEUTRAL.",
        ... )
        >>> "NEGATIVE" in prompt or "Classify" in prompt
        True
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _run_checks():
    score, total = 0, 4

    # Check 1: function is defined and callable
    try:
        assert callable(build_prompt), 'build_prompt is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return  # cannot continue — later checks would crash

    # Check 2: return type is str
    try:
        result = build_prompt(
            instruction="Summarise this.",
            context="You are a helpful assistant.",
            user_input="The sky is blue.",
            output_format="One sentence.",
        )
        assert isinstance(result, str), f'expected str, got {type(result).__name__}'
        print(f'{_PASS} Check 2/{total}: return type is str')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return  # str checks below would throw AttributeError on None

    # Check 3: all four parts appear in the returned string
    try:
        instruction   = "Classify the sentiment of this review."
        context       = "You are a sentiment analyser for a retail company."
        user_input    = "The battery life on this phone is absolutely terrible."
        output_format = "Reply with exactly one word: POSITIVE, NEGATIVE, or NEUTRAL."

        prompt = build_prompt(instruction, context, user_input, output_format)

        missing = []
        for part, label in [
            (instruction,   'instruction'),
            (context,       'context'),
            (user_input,    'user_input'),
            (output_format, 'output_format'),
        ]:
            if part not in prompt:
                missing.append(label)

        assert not missing, f'missing parts: {missing}'
        print(f'{_PASS} Check 3/{total}: all four parts present in the assembled prompt')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: parts appear in the conventional order (instruction before context
    #          before input before format) so the prompt reads coherently
    try:
        instruction   = "Extract the product name."
        context       = "You are a product data extractor."
        user_input    = "I just bought the UltraBlend 3000 and love it."
        output_format = "Reply with only the product name, nothing else."

        prompt = build_prompt(instruction, context, user_input, output_format)

        idx_instruction   = prompt.index(instruction)
        idx_context       = prompt.index(context)
        idx_input         = prompt.index(user_input)
        idx_output_format = prompt.index(output_format)

        assert idx_instruction <= idx_context, \
            'instruction should appear before or at context'
        assert idx_context <= idx_input, \
            'context should appear before or at input'
        assert idx_input <= idx_output_format, \
            'input should appear before or at output_format'

        print(f'{_PASS} Check 4/{total}: parts are ordered instruction → context → input → output_format')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {score}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

In production pipelines the four parts are often split across two message roles: `system` (instruction + context) and `user` (input + output format). This maps onto the role-based API you used in Day 3.

Extend `build_prompt` — or write a new function `build_messages` — that returns a `list[dict]` of `{"role": ..., "content": ...}` message objects, ready to drop straight into `ollama.chat(model=..., messages=...)`.

This is exactly the pattern Lesson 2 and the Day 6 project rely on.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def build_prompt(instruction: str, context: str, user_input: str, output_format: str) -> str:
    return (
        f"{instruction}\n\n"
        f"{context}\n\n"
        f"Input:\n{user_input}\n\n"
        f"{output_format}"
    )
```

**Why this works:** each part is placed on its own labelled block separated by blank lines, which mirrors how the model was trained to read structured instructions — the blank lines act as visual section breaks that the model uses to separate the task definition from the background from the data. Keeping the four parts as distinct variables in your code (rather than one big f-string) means that when a prompt gives a wrong result you can point at exactly which variable to fix, turning vague prompt debugging into a one-variable diff.
</details>